# 02 — Median Baseline (cross-project)

Predicts the **source project's training-median story point** for every issue in the **target project's test set**.
Runs every ordered source→target pair (39×38 = 1,482). Your 39-pair table (PDF) is a subset — filter later.

Also computes the within-project baseline (source = target) for reference.

Output: `baseline_results.csv` (per pair: MAE, MdAE, n_train, n_test) + `baseline_summary.txt`.
Input: kernel output of `srinithetta/tawos-extract` (all_issues_sp.csv). CPU only.

In [ ]:
import glob, os
import numpy as np, pandas as pd

hits = glob.glob("/kaggle/input/**/all_issues_sp.csv", recursive=True)
assert hits, "attach the tawos-extract kernel output as Input"
DF = hits[0]
df = pd.read_csv(DF)
df['storypoint'] = pd.to_numeric(df['storypoint'], errors='coerce')
df = df.dropna(subset=['storypoint'])

# outlier policy (documented): drop SP=0 and SP>40 — garbage/no-effort tickets; matches common TAWOS filtering
n_raw = len(df)
df = df[(df.storypoint > 0) & (df.storypoint <= 40)]
print(f"{n_raw} -> {len(df)} rows after SP in (0,40] filter")

SEED = 42
rng = np.random.default_rng(SEED)

# fixed per-project 80:20 split: train pool (for median) and test set
splits = {}
for proj, g in df.groupby('project'):
    idx = g.index.to_numpy()
    rng.shuffle(idx)
    n_test = max(1, int(round(0.2 * len(idx))))
    splits[proj] = dict(test=idx[:n_test], train=idx[n_test:])

projects = sorted(splits)
medians = {p: float(df.loc[splits[p]['train'], 'storypoint'].median()) for p in projects}
print(len(projects), "projects | example medians:", {p: medians[p] for p in projects[:5]})

In [ ]:
rows = []
for src in projects:
    med = medians[src]
    for tgt in projects:
        if tgt == src:
            continue
        y = df.loc[splits[tgt]['test'], 'storypoint'].to_numpy()
        err = np.abs(y - med)
        rows.append(dict(source=src, target=tgt,
                         mae=float(err.mean()), mdae=float(np.median(err)),
                         pred=med, n_train=len(splits[src]['train']), n_test=len(y)))

# within-project reference
for p in projects:
    y = df.loc[splits[p]['test'], 'storypoint'].to_numpy()
    err = np.abs(y - medians[p])
    rows.append(dict(source=p, target=p + ' [WITHIN]',
                     mae=float(err.mean()), mdae=float(np.median(err)),
                     pred=medians[p], n_train=len(splits[p]['train']), n_test=len(y)))

res = pd.DataFrame(rows).sort_values(['source', 'target'])
res.to_csv('/kaggle/working/baseline_results.csv', index=False)
cross = res[res.source != res.target.str.replace(' [WITHIN]', '', regex=False)] if False else res[~res.target.str.contains('WITHIN')]
within = res[res.target.str.contains('WITHIN')]

summary = [
    f"seed {SEED} | filter SP in (0,40] | {len(df)} issues | {len(projects)} projects",
    f"CROSS-PROJECT  (n={len(cross)} pairs): MAE mean {cross.mae.mean():.3f} +- {cross.mae.std():.3f} | MdAE mean {cross.mdae.mean():.3f}",
    f"WITHIN-PROJECT (n={len(within)}):      MAE mean {within.mae.mean():.3f} +- {within.mae.std():.3f} | MdAE mean {within.mdae.mean():.3f}",
]
txt = chr(10).join(summary)
open('/kaggle/working/baseline_summary.txt', 'w').write(txt)
print(txt)
print("\nbest 5 cross pairs:")
print(cross.nsmallest(5, 'mae')[['source', 'target', 'mae', 'mdae']].to_string(index=False))
print("\nworst 5 cross pairs:")
print(cross.nlargest(5, 'mae')[['source', 'target', 'mae', 'mdae']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(18, 5))
cross.groupby('source').mae.mean().sort_values().plot.bar(ax=ax, color='steelblue')
ax.axhline(cross.mae.mean(), color='red', ls='--', label=f"overall {cross.mae.mean():.2f}")
ax.set_title('Median-baseline MAE by SOURCE project (lower = easier source)')
ax.set_ylabel('MAE')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/baseline_by_source.png', dpi=120)
plt.show()

## Reading the numbers
- These MAEs are the **floor** every BERT+CNN result must beat, per pair.
- Within-project vs cross-project gap = the difficulty tax of your PS-3 setting — quote both in the report.
- `pred` column = the source's median; wildly different medians across projects explain bad pairs (scale mismatch) before any model is trained.